# TPCDrum Event-level AI Baseline (Notebook)

이 노트북은 `tpcd_ai_event.csv`를 사용해 `true_dxdy` 또는 `true_dzdy`를 예측합니다.

- 입력 feature: event-level summary columns
- 타깃: `true_dxdy` 또는 `true_dzdy`
- 모델: `RandomForestRegressor`
- 주의: RMSE는 `np.sqrt(mean_squared_error(...))`로 계산


In [ ]:
# (최초 1회) 패키지 설치
# !pip install -r requirements.txt


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split


In [ ]:
# ===== 사용자 설정 =====
CSV_PATH = '../../tpcd_ai_event.csv'   # 노트북 위치: project/starter/
TARGET = 'true_dxdy'                   # 또는 'true_dzdy'
SEED = 42
OUTDIR = Path('../../outputs_notebook_dxdy')
OUTDIR.mkdir(parents=True, exist_ok=True)


In [ ]:
df = pd.read_csv(CSV_PATH)
print('shape =', df.shape)
df.head(3)


In [ ]:
drop_cols = {'event', 'true_dxdy', 'true_dzdy'}
features = [c for c in df.columns if c not in drop_cols]

X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0.0)
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)

model = RandomForestRegressor(
    n_estimators=400,
    max_depth=None,
    random_state=SEED,
    n_jobs=-1,
)
model.fit(X_train, y_train)
pred = model.predict(X_test)

rmse = float(np.sqrt(mean_squared_error(y_test, pred)))
mae = float(mean_absolute_error(y_test, pred))
r2 = float(r2_score(y_test, pred))

print(f'target={TARGET}')
print(f'n_events={len(df)}')
print(f'n_features={len(features)}')
print(f'RMSE={rmse:.6f}')
print(f'MAE={mae:.6f}')
print(f'R2={r2:.6f}')


In [ ]:
fi = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
fi.head(15)


In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(y_test, pred, s=15, alpha=0.7)
lo = min(y_test.min(), pred.min())
hi = max(y_test.max(), pred.max())
plt.plot([lo, hi], [lo, hi], 'r--', lw=1)
plt.xlabel(f'Truth ({TARGET})')
plt.ylabel('Prediction')
plt.title(f'AI vs Truth ({TARGET})')
plt.tight_layout()
plt.show()


In [ ]:
# 결과 저장
fi.to_csv(OUTDIR / 'feature_importance.csv', index=True)
metrics_txt = (
    f'target={TARGET}\n'
    f'n_events={len(df)}\n'
    f'n_features={len(features)}\n'
    f'RMSE={rmse:.6f}\n'
    f'MAE={mae:.6f}\n'
    f'R2={r2:.6f}\n'
)
(OUTDIR / 'metrics.txt').write_text(metrics_txt, encoding='utf-8')

print('saved:', OUTDIR)
